# Agent的高级用法-错误处理机制
## 设置True/False/固定字符串
### handle_errors设置为True

In [1]:
import os
from typing import Literal
from dotenv import load_dotenv
from langchain_core.tools import tool

from langchain.agents.structured_output import AutoStrategy,ToolStrategy 
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_protocol import Union
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)
ARK_API_KEY = os.getenv("ARK_API_KEY")
ARK_MODEL_NAME = os.getenv("ARK_MODEL_NAME")
ARK_BASE_URL = os.getenv("ARK_BASE_URL")
model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)
#使用pydantic结构化方法进行定义

class ContraactInfo(BaseModel):
    '''用户联系方式'''
    name : str = Field(description="用户姓名")
    email : str = Field(description="用户邮箱")
    phone : str = Field(description="用户电话")


# 简化toolMessage的内容
from langgraph.prebuilt import tools_condition


# 定义第一个 Schema：产品评论
class ProductReview(BaseModel):
    """关于产品的评论"""
    rating: int | None = Field(description="产品评分 (1-5)")
    sentiment: Literal["positive", "negative"] = Field(description="评论情感")
    key_points: list[str] = Field(description="评论要点")

# 定义第二个 Schema：客户投诉
class CustomerComplaint(BaseModel):
    """关于产品或服务的客户投诉"""
    issue_type: Literal["product", "service", "shipping", "billing"] = Field(description="问题类型")
    severity: Literal["low", "medium", "high"] = Field(description="严重程度")
    description: str = Field(description="投诉简述")

agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功"
        ),
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复购'"]
})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

c:\Users\XSH\.conda\envs\langchain1.2\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


================================ Human Message =================================

分析这条评论：'好用，爱用，还会再次复购'
================================== Ai Message ==================================

我来分析这条评论。
Tool Calls:
  ProductReview (call_knuxslyieygo7taszsn6xdva)
 Call ID: call_knuxslyieygo7taszsn6xdva
  Args:
    key_points: ['好用 - 产品使用体验良好', '爱用 - 用户对产品有明显的喜爱', '还会再次复购 - 表明用户满意度高，有复购意向和忠诚度']
    rating: 5
    sentiment: positive
================================= Tool Message =================================
Name: ProductReview

评价分析成功


ProductReview(
    rating=5,
    sentiment='positive',
    key_points=[
        '好用 - 产品使用体验良好',
        '爱用 - 用户对产品有明显的喜爱',
        '还会再次复购 - 表明用户满意度高，有复购意向和忠诚度'
    ]
)

默认handle_errors 的值为true，上述请求，我们大模型会去匹配两个对象但是unio默认只输出一个对象，我们也可以看到，上述输出中包含warning提示。打印完整的respnse我们看一下，之后我们尝试关闭 handle_errors

In [2]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content="分析这条评论：'好用，爱用，还会再次复购'",
            additional_kwargs={},
            response_metadata={},
            id='c4383ecc-5505-4ff7-97c6-a4c325631a4c'
        ),
        AIMessage(
            content='我来分析这条评论。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 199,
                    'prompt_tokens': 371,
                    'total_tokens': 570,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 131,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179051959832391be90b70b1bfc3059428bd07e108b7e3cf8bc',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e348-dcd0-74c2-9d16-39e0486129c4-0',
            tool_calls=[
                {
                    'name': 'ProductReview',
                    'args': {
                        'key_points': [
                            '好用 - 产品使用体验良好',
                            '爱用 - 用户对产品有明显的喜爱',
                            '还会再次复购 - 表明用户满意度高，有复购意向和忠诚度'
                        ],
                        'rating': '5',
                        'sentiment': 'positive'
                    },
                    'id': 'call_knuxslyieygo7taszsn6xdva',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 371,
                'output_tokens': 199,
                'total_tokens': 570,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 131}
            }
        ),
        ToolMessage(
            content='评价分析成功',
            name='ProductReview',
            id='77e9c8d8-3984-459b-a9d1-266c61e83438',
            tool_call_id='call_knuxslyieygo7taszsn6xdva'
        )
    ],
    'structured_response': ProductReview(
        rating=5,
        sentiment='positive',
        key_points=[
            '好用 - 产品使用体验良好',
            '爱用 - 用户对产品有明显的喜爱',
            '还会再次复购 - 表明用户满意度高，有复购意向和忠诚度'
        ]
    )
}

好像不太对劲，上面的warning是 requests 库在导入时发现没有安装字符检测依赖（chardet 或 charset_normalizer）而发出的警告我们换一个对象看看包含错误信息的toolmessages

In [5]:
from pydantic import BaseModel, Field, field_validator

class ProductEvaluation(BaseModel):
    product_name: str = Field(default="", description="产品名称")
    rating: int = Field(default=1, description="评分1-5", ge=1, le=5)
    sentiment: Literal["正面", "负面", "中性"] = Field(default="中性")

    @field_validator("rating")
    def rating_must_be_three(cls, v):
        if v != 3:
            raise ValueError("rating 必须等于 3")  # 隐藏约束，schema 描述里看不出
        return v
resp = agent.invoke({"messages": [{"role": "user", "content": "这个产品很不错，我给满分10分"}]})

for m in resp["messages"]:
    if m.type == "tool":
        print(f"[tool] content = {m.content}")

[tool] content = 评价分析成功


In [6]:
rprint(resp)

{
    'messages': [
        HumanMessage(
            content='这个产品很不错，我给满分10分',
            additional_kwargs={},
            response_metadata={},
            id='0efc655d-3589-4ea6-8295-9adee87d02f9'
        ),
        AIMessage(
            content='您好！感谢您对产品的积极反馈。我将您给出的10分满分转换为我们系统的5分制评分（即5分满分）进行记
录：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 193,
                    'prompt_tokens': 365,
                    'total_tokens': 558,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 114,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '0217905201232639255aa03547266d00c3fa3dce78057f1fe3117',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e350-df51-7fa3-9a44-4e7ba7a682e8-0',
            tool_calls=[
                {
                    'name': 'ProductReview',
                    'args': {
                        'key_points': ['用户对产品非常满意', '给出满分评价（10分制，折合5分制满分）'],
                        'rating': '5',
                        'sentiment': 'positive'
                    },
                    'id': 'call_gz6fd89hf6auyhac5zdd27a2',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 365,
                'output_tokens': 193,
                'total_tokens': 558,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 114}
            }
        ),
        ToolMessage(
            content='评价分析成功',
            name='ProductReview',
            id='80a677ac-4afe-46f2-924b-c66e5e14e22c',
            tool_call_id='call_gz6fd89hf6auyhac5zdd27a2'
        )
    ],
    'structured_response': ProductReview(
        rating=5,
        sentiment='positive',
        key_points=['用户对产品非常满意', '给出满分评价（10分制，折合5分制满分）']
    )
}

In [7]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class ProductReview(BaseModel):
    """关于产品的评论"""
    rating: int | None = Field(default=None, description="产品评分 (1-5)")
    sentiment: Literal["positive", "negative"] = Field(description="评论情感")
    key_points: list[str] = Field(description="评论要点")

    @field_validator("rating")
    @classmethod
    def rating_must_be_three(cls, v):
        # 这个约束不写在 description 里，模型看不到
        if v != 3:
            raise ValueError("rating 必须恰好等于 3（内部规则）")
        return v
resp = agent.invoke({"messages": [{"role": "user", "content": "这个产品很不错，我给满分10分"}]})

for m in resp["messages"]:
    if m.type == "tool":
        print(f"[tool] content = {m.content}")
rprint(resp)

[tool] content = 评价分析成功


{
    'messages': [
        HumanMessage(
            content='这个产品很不错，我给满分10分',
            additional_kwargs={},
            response_metadata={},
            id='5d40532d-11a6-4994-9801-23a0b24a6f23'
        ),
        AIMessage(
            content='收到您的评价！我们的评价系统采用 1-5 分制，您给的满分 10 分将记录为 5 分满分好评。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 380,
                    'prompt_tokens': 365,
                    'total_tokens': 745,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 308,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052021947278af8113c224ccdd69578c3b63d1e71aec7cb7',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e352-5724-7221-84b8-a59dc2482665-0',
            tool_calls=[
                {
                    'name': 'ProductReview',
                    'args': {
                        'key_points': ['产品很不错', '满分好评（10分制折合5分满分）'],
                        'rating': '5',
                        'sentiment': 'positive'
                    },
                    'id': 'call_d56ap4ry0yqh4copqirxbkmy',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 365,
                'output_tokens': 380,
                'total_tokens': 745,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 308}
            }
        ),
        ToolMessage(
            content='评价分析成功',
            name='ProductReview',
            id='7a5d69ea-8802-4c01-9a47-e8b88aeb72e2',
            tool_call_id='call_d56ap4ry0yqh4copqirxbkmy'
        )
    ],
    'structured_response': ProductReview(
        rating=5,
        sentiment='positive',
        key_points=['产品很不错', '满分好评（10分制折合5分满分）']
    )
}

放弃了，说是模型的问题不会触发ToolStrategy 的校验重试机制。我们直接试试=false

In [ ]:
import os
from typing import Literal
from dotenv import load_dotenv
from langchain_core.tools import tool

from langchain.agents.structured_output import AutoStrategy,ToolStrategy 
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_protocol import Union
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)
ARK_API_KEY = os.getenv("ARK_API_KEY")
ARK_MODEL_NAME = os.getenv("ARK_MODEL_NAME")
ARK_BASE_URL = os.getenv("ARK_BASE_URL")
model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)
#使用pydantic结构化方法进行定义

class ContraactInfo(BaseModel):
    '''用户联系方式'''
    name : str = Field(description="用户姓名")
    email : str = Field(description="用户邮箱")
    phone : str = Field(description="用户电话")


# 简化toolMessage的内容
from langgraph.prebuilt import tools_condition


# 定义第一个 Schema：产品评论
class ProductReview(BaseModel):
    """关于产品的评论"""
    rating: int | None = Field(description="产品评分 (1-5)")
    sentiment: Literal["positive", "negative"] = Field(description="评论情感")
    key_points: list[str] = Field(description="评论要点")

# 定义第二个 Schema：客户投诉
class CustomerComplaint(BaseModel):
    """关于产品或服务的客户投诉"""
    issue_type: Literal["product", "service", "shipping", "billing"] = Field(description="问题类型")
    severity: Literal["low", "medium", "high"] = Field(description="严重程度")
    description: str = Field(description="投诉简述")

agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors= False
        ),
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复购'"]
})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

================================ Human Message =================================

分析这条评论：'好用，爱用，还会再次复购'
================================== Ai Message ==================================

我来分析这条评论的情感倾向和关键信息。
Tool Calls:
  ProductReview (call_ogpfzvte6por9i7a1m63z6pv)
 Call ID: call_ogpfzvte6por9i7a1m63z6pv
  Args:
    key_points: ['产品好用，使用体验良好', '用户表示喜爱该产品', '有明确的再次复购意愿']
    rating: 5
    sentiment: positive
================================= Tool Message =================================
Name: ProductReview

评价分析成功


ProductReview(
    rating=5,
    sentiment='positive',
    key_points=['产品好用，使用体验良好', '用户表示喜爱该产品', '有明确的再次复购意愿']
)

返回固定字符串 
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors= "请检查输入字符串"
        ),
模型太智能了，我们试着主动触发错误看看。


In [9]:
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功"
        ),
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

================================ Human Message =================================

分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'
================================== Ai Message ==================================

我来分析这条评论。这条评论整体是正面评价，但包含一个关于客服响应速度的小建议，我将分别记录产品评论和客户反馈：
Tool Calls:
  ProductReview (call_16k6nfch30xfbi8zlg860pm6)
 Call ID: call_16k6nfch30xfbi8zlg860pm6
  Args:
    key_points: ['产品好用', '用户很喜欢', '表示会再次复购', '希望客服回复速度更快一些']
    rating: 4
    sentiment: positive
  CustomerComplaint (call_4aa5vgqnlgv4mcz082ozj3dm)
 Call ID: call_4aa5vgqnlgv4mcz082ozj3dm
  Args:
    description: 客服回复速度偏慢，客户希望能更快得到响应（非严重问题，客户整体仍满意产品）
    issue_type: service
    severity: low
================================= Tool Message =================================
Name: ProductReview

Error: Model incorrectly returned multiple structured responses (ProductReview, CustomerComplaint) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name

ProductReview(
    rating=4,
    sentiment='positive',
    key_points=['产品好用', '用户很喜欢', '表示会再次复购', '建议：客服回复速度能再快一点就更好了']
)

看来不是太智能了，是设置的问题，没有设置能够触发错误的问题。再看看完整回复内容

In [10]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content="分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'",
            additional_kwargs={},
            response_metadata={},
            id='49dce44e-c0d6-41f0-9551-982358a9d25c'
        ),
        AIMessage(
            content='我来分析这条评论。这条评论整体是正面评价，但包含一个关于客服响应速度的小建议，我将分别记录产品
评论和客户反馈：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 295,
                    'prompt_tokens': 379,
                    'total_tokens': 674,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 172,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052067698889d0c17fc196b3c940c12dd7ad5b500ba1a369',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e359-5255-7a92-8430-bb51d238fd7b-0',
            tool_calls=[
                {
                    'name': 'ProductReview',
                    'args': {
                        'key_points': ['产品好用', '用户很喜欢', '表示会再次复购', '希望客服回复速度更快一些'],
                        'rating': '4',
                        'sentiment': 'positive'
                    },
                    'id': 'call_16k6nfch30xfbi8zlg860pm6',
                    'type': 'tool_call'
                },
                {
                    'name': 'CustomerComplaint',
                    'args': {
                        'description': 
'客服回复速度偏慢，客户希望能更快得到响应（非严重问题，客户整体仍满意产品）',
                        'issue_type': 'service',
                        'severity': 'low'
                    },
                    'id': 'call_4aa5vgqnlgv4mcz082ozj3dm',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 379,
                'output_tokens': 295,
                'total_tokens': 674,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 172}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ProductReview, 
CustomerComplaint) when only one is expected.\n Please fix your mistakes.',
            name='ProductReview',
            id='a3b8b8ef-fef3-4030-8a2d-715e59c4c734',
            tool_call_id='call_16k6nfch30xfbi8zlg860pm6'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ProductReview, 
CustomerComplaint) when only one is expected.\n Please fix your mistakes.',
            name='CustomerComplaint',
            id='53a2c8f7-0d4a-434a-944d-4dd884047a91',
            tool_call_id='call_4aa5vgqnlgv4mcz082ozj3dm'
        ),
        AIMessage(
            content='看来系统一次只接受一条结构化回复。评论的核心是对产品的评价，我将把客服响应速度的建议作为评论要
点一并记录：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 131,
                    'prompt_tokens': 563,
                    'total_tokens': 694,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
        

ok,接下来把错误处理关闭看看handle_errors= False

In [11]:
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors= False
        ),
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ProductReview, CustomerComplaint) when only one is expected.

ok,也是成功触发报错，MultipleStructuredOutputsError，这个应该就是之前我们预期的多个模型匹配的错误，接下来试试看字符串的内容。统一回复handle_errors="不好意思，我不太清楚您的意思TAT"

In [12]:
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors="不好意思，我不太清楚您的意思TAT"
        ),
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

KeyboardInterrupt: 

有问题，大模型直接死机了，内部循环调用，先跳过把

In [16]:
model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors="不好意思，我不太清楚您的意思TAT"
        )
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
},
    config={"recursion_limit":3})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

GraphRecursionError: Recursion limit of 3 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

加上接口限制，貌似还是不太行

In [17]:
from langchain.agents.structured_output import MultipleStructuredOutputsError,StructuredOutputValidationError

model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
        ) 
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
},
    config={"recursion_limit":3})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

================================ Human Message =================================

分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'
================================== Ai Message ==================================

这条评论包含正面产品评价和轻微的服务建议，我将分别记录产品评论和客服反馈：
Tool Calls:
  ProductReview (call_9d7z5xgu9gcj8qt8llj8pxnp)
 Call ID: call_9d7z5xgu9gcj8qt8llj8pxnp
  Args:
    key_points: ['产品好用', '用户喜爱该产品', '表示会再次复购', '希望客服回复速度能更快一些']
    rating: 4
    sentiment: positive
  CustomerComplaint (call_cgl87vfjq5xae71j6nha8juy)
 Call ID: call_cgl87vfjq5xae71j6nha8juy
  Args:
    description: 客户反映客服回复速度较慢，希望能加快响应速度。属于轻微建议，客户对产品本身非常满意并愿意复购。
    issue_type: service
    severity: low
================================= Tool Message =================================
Name: ProductReview

Error: Model incorrectly returned multiple structured responses (ProductReview, CustomerComplaint) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: Cus

ProductReview(
    rating=4,
    sentiment='positive',
    key_points=['产品好用，用户喜爱', '明确表示会再次复购', '建议客服回复速度能更快一些']
)

In [18]:
from langchain.agents.structured_output import MultipleStructuredOutputsError,StructuredOutputValidationError

model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)
agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        # handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
        handle_errors=(StructuredOutputValidationError)
        ) 
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
},
    config={"recursion_limit":3})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ProductReview, CustomerComplaint) when only one is expected.

监听特定的错误MultipleStructuredOutputsError和StructuredOutputValidationError

接下来试试自定义错误

In [ ]:
from langchain.agents.structured_output import MultipleStructuredOutputsError,StructuredOutputValidationError

def custiom_error_handler(error : Exception) -> str:
    """自定义错误处理器"""
    error_str =  str(error)

    print(f"捕获到的错误类型是：{type(error).__name__}")
    print(f"错误详情：{error_str}")

    if isinstance(error,MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个返回"
    elif isinstance(error,StructuredOutputValidationError):
        return "数据格式有误，请检查输入数据"
    else:
        return f"Error:{error_str}" 

agent = create_agent(
    model=model,  # 示例模型
    response_format=ToolStrategy(
        schema= Union[ProductReview, CustomerComplaint],
        tool_message_content= "评价分析成功",
        handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
        ) 
    
)
result = agent.invoke({
    "messages": ["分析这条评论：'好用，爱用，还会再次复，要是客服能回复再快一点就好了'"]
},
    config={"recursion_limit":3})

for message in result["messages"]:
    message.pretty_print()
rprint(result["structured_response"])